# Lesson 6.2 — Detecting Failure: Health Signals and Guards

A guard at each seam reads an existing output and fires the matching event. Hard failures halt the path at their stage; a healthy run reaches Track.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, run_pipeline, localize, failure_event,
    FAILURE_TAXONOMY, REACH_MAX)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def big(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []


### Each hard fault halts at its own seam

In [2]:
# NO_TARGET at Understand
r1 = run_pipeline(W, W.q.copy(), perception=dict(occlude=ALL_IDS))
print('occlude     -> reached=%-10s event=%s' % (r1['reached'], [e['code'] for e in r1['events']]))
checks.append(r1['reached']=='Understand' and r1['events'][0]['code']=='NO_TARGET')
# PLAN_INVALID at Plan (obstacle over the goal)
r2 = run_pipeline(W, W.q.copy(), obstacle=(TGT['xy'], 0.25))
print('block goal  -> reached=%-10s event=%s' % (r2['reached'], [e['code'] for e in r2['events']]))
checks.append(r2['reached']=='Plan' and r2['events'][0]['code']=='PLAN_INVALID')
# TRACKING_FAILURE at Track (disturbance)
r3 = run_pipeline(W, W.q.copy(), disturbance=big)
print('disturbance -> reached=%-10s event=%s' % (r3['reached'], [e['code'] for e in r3['events']]))
checks.append(r3['reached']=='Track' and any(e['code']=='TRACKING_FAILURE' for e in r3['events']))


occlude     -> reached=Understand event=['NO_TARGET']


block goal  -> reached=Plan       event=['PLAN_INVALID']


disturbance -> reached=Track      event=['TRACKING_FAILURE', 'EXCESS_EFFORT']


### A healthy run passes every guard

In [3]:
r0 = run_pipeline(W, W.q.copy())
print('healthy     -> reached=%-10s success=%s events=%s' % (r0['reached'], r0['success'], [e['code'] for e in r0['events']]))
checks.append(r0['reached']=='Track' and r0['success'] and r0['events']==[])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


healthy     -> reached=Track      success=True events=[]
4/4 checks passed.
All checks passed.
